## Producer C — Kafka Event Stream Initialisation

This cell initialises and runs **Producer C**, which reads camera events from `camera_event_C.csv`
and publishes them to the Kafka topic `camera-events-C` in batches grouped by `batch_id`.

### Key Parameters

| Parameter | Value | Rationale |
|---|---|---|
| `bootstrap_servers` | `kafka:9092` | Kafka broker address as defined in Docker Compose network. Using the service name `kafka` instead of `localhost` ensures correct container-to-container routing. |
| `topic` | `camera-events-C` | Dedicated topic per producer isolates each camera stream, allowing the Spark consumer to subscribe selectively and apply per-stream logic. |
| `camera_id` | `3` | Identifies the source camera for all events published by this producer, enabling traceability in the downstream violation detection logic. |
| `batch_interval` | `5 seconds` | Events are grouped by `batch_id` and published one batch every 5 seconds. This pacing was confirmed as appropriate by the teaching team. It provides a realistic simulation of a camera emitting periodic snapshots, while giving Spark sufficient time to process each micro-batch before the next arrives. |
| `csv_path` | `../data/camera_event_C.csv` | Relative path to the camera event dataset for Producer C, following the submission directory structure defined in the specification. |

### Execution Notes

- The producer runs continuously until manually interrupted (`KeyboardInterrupt`).
- On interrupt or error, `producer_c.close()` is called in the `finally` block to ensure the Kafka
  producer is gracefully shut down and all buffered messages are flushed.
- Run this notebook **concurrently** with Producer A and Producer B notebooks to simulate
  simultaneous multi-camera event ingestion, as required by the streaming join logic in Task 2.1.2.

In [ ]:
from pathlib import Path

from camera_event_producer import CameraEventProducer

HOST_IP = "kafka"  # Docker Compose service name
csv_path = Path("..") / "data" / "camera_event_C.csv"
producer_c = CameraEventProducer(
    bootstrap_servers=[f"{HOST_IP}:9092"],
    topic="camera-events-C",
    camera_id=3,
    csv_path=str(csv_path),
    batch_interval=5
)

try:
    producer_c.publish_batches()
except KeyboardInterrupt:
    print("Producer C stopped by user")
finally:
    producer_c.close()

[2026-05-25T14:54:22.543593] Published batch 1 to camera-events-C (1 events)
[2026-05-25T14:54:27.563193] Published batch 2 to camera-events-C (1 events)
[2026-05-25T14:54:32.573565] Published batch 3 to camera-events-C (2 events)
[2026-05-25T14:54:37.583560] Published batch 4 to camera-events-C (2 events)
[2026-05-25T14:54:42.594597] Published batch 5 to camera-events-C (1 events)
[2026-05-25T14:54:47.603128] Published batch 6 to camera-events-C (1 events)
[2026-05-25T14:54:52.610945] Published batch 7 to camera-events-C (2 events)
[2026-05-25T14:54:57.621889] Published batch 8 to camera-events-C (1 events)
[2026-05-25T14:55:02.627686] Published batch 9 to camera-events-C (2 events)
[2026-05-25T14:55:07.639008] Published batch 10 to camera-events-C (2 events)
[2026-05-25T14:55:12.647017] Published batch 11 to camera-events-C (1 events)
[2026-05-25T14:55:17.658361] Published batch 12 to camera-events-C (1 events)
[2026-05-25T14:55:22.665484] Published batch 13 to camera-events-C (1 eve